# End-to-End Unsupervised Pipeline — Text Clustering (Assignment)

## Goal
Take **unlabeled text** and discover groups in it. You'll use a 4-topic subset of the 20 Newsgroups dataset — labels exist (so you can sanity-check) but you'll cluster *without using them*.

Each step tells you *what* to implement and *why*. Write the code yourself in the empty cells. When stuck, check `03_unsupervised_end_to_end.ipynb` for the worked solution.

## Dataset
`sklearn.datasets.fetch_20newsgroups` with categories `['rec.sport.baseball', 'sci.space', 'comp.graphics', 'talk.politics.mideast']`. Strip headers/footers/quotes — they leak signal.

## Deliverable
- `artifacts_unsupervised/cluster_model.pkl` and `.joblib`
- `artifacts_unsupervised/metadata.json` — versions, metrics, top terms per cluster
- A working reload that assigns cluster ids to brand-new sentences.

---

## Step 0: Imports

All libraries you'll need are pre-loaded below — just run the cell.

In [ ]:
import os, re, json, pickle, joblib, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import Normalizer, normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.pipeline import Pipeline
import sklearn
import warnings
warnings.filterwarnings('ignore')

# Optional NLP toolkit — used in Step 4 if available
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    from nltk.corpus import stopwords as nltk_stop
    from nltk.stem import WordNetLemmatizer
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42
print('Imports OK  |  NLTK available:', HAS_NLTK)

## Step 1: Define the problem

**Task:** in markdown, answer:
1. What does a "good" cluster look like (cohesion vs separation)?
2. How will you validate quality without labels in a real scenario?
3. Real-world parallels (customer support tickets, news, product reviews)?
4. Since this dataset HAS labels, how will you double-check your clusters?

**YOUR ANSWERS**
1. 
2. 
3. 
4. 

## Step 2: Load the data

Dataset is pre-loaded below. Just run the cell.

In [ ]:
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics', 'talk.politics.mideast']
data = fetch_20newsgroups(subset='all', categories=categories,
                          remove=('headers', 'footers', 'quotes'),
                          random_state=RANDOM_STATE)

df = pd.DataFrame({'text': data.data, 'true_label': data.target})
df['true_topic'] = df['true_label'].map({i: c for i, c in enumerate(data.target_names)})
df = df[df['text'].str.len() > 50].reset_index(drop=True)
print(f'Documents: {len(df)}')
print(df['true_topic'].value_counts())

## Step 3: EDA

**Task:**
1. Add `n_chars` and `n_words` columns. Print `.describe()`.
2. Plot histogram of words per doc (clip to 1500 to ignore outliers).
3. Plot median words per topic as a bar chart.
4. Print one sample document — read it.

In [ ]:
# YOUR CODE HERE


## Step 4: NLP preprocessing

**Task:** write `clean_text(t)` that does:
1. Lowercase.
2. Strip emails (`\S+@\S+`), URLs, all digit runs.
3. Strip non-letters except spaces.
4. Collapse whitespace.
5. Drop stopwords AND tokens with length ≤ 2.
6. Lemmatize with `WordNetLemmatizer` if NLTK is available, else just return the cleaned tokens.

Then wrap it in a `TextCleaner(BaseEstimator, TransformerMixin)`.

**Why aggressive cleaning for clustering?** Distance is computed on every TF-IDF dimension. Noise tokens make distances meaningless. Vocabulary pruning is unsupervised's biggest accuracy lever.

In [ ]:
# YOUR CODE HERE


## Step 5: Vectorize with TF-IDF

**Task:** apply your cleaner to all docs, then fit a `TfidfVectorizer` with:
- `ngram_range=(1, 2)`
- `min_df=5` (drop super-rare terms — noise)
- `max_df=0.5` (drop terms that appear in >50% of docs — too generic)
- `sublinear_tf=True` (dampen high counts)

Print the resulting matrix shape (`docs × terms`).

In [ ]:
# YOUR CODE HERE


## Step 6: Reduce dimensions with TruncatedSVD

**Task:** apply `TruncatedSVD(n_components=100)` then L2-`normalize` the result. Print the resulting shape and total variance explained.

**Why?** Distance is meaningless in thousands of sparse dimensions. SVD on TF-IDF (called LSA) projects to a dense low-dim space where semantically similar docs end up close. Normalizing makes cosine similarity work like Euclidean distance — so plain K-Means makes sense.

In [ ]:
# YOUR CODE HERE


## Step 7: Choose K — elbow + silhouette

**Task:** for K in 2..10, fit K-Means and record `inertia_` and `silhouette_score` (use `sample_size=2000` to keep it fast). Plot both. Pick K and assign it to a variable `K`.

**Hint:** because we know there are 4 true topics, you should expect the elbow around K=4.

In [ ]:
# YOUR CODE HERE


## Step 8: Try multiple algorithms

**Task:** run all three on the SVD-reduced data:
- `KMeans(n_clusters=K)`
- `AgglomerativeClustering(n_clusters=K, linkage='ward')`
- `DBSCAN(eps=0.5, min_samples=10, metric='cosine')`

For each, compute: number of clusters found, noise count (DBSCAN only), silhouette, ARI vs `true_label`, NMI vs `true_label`. Print as a comparison table.

**Why ARI/NMI?** They measure how well your clusters match the held-out true topics. In a real unsupervised problem you wouldn't have these — you'd rely on silhouette + reading samples.

In [ ]:
# YOUR CODE HERE


## Step 9: Inspect clusters — top terms + sample documents

**Task:** for the K-Means result:
1. Use `svd.inverse_transform(km.cluster_centers_)` to project centroids back to TF-IDF terms.
2. For each cluster, print the top 12 highest-weighted terms.
3. Print 2 sample documents from each cluster.
4. Build a `pd.crosstab(true_topic, cluster)` and print it.

**Why:** this is the unsupervised equivalent of "reading misclassified examples". A good cluster has top terms that tell a coherent story and sample docs that are clearly on-topic.

In [ ]:
# Top terms per cluster
# YOUR CODE HERE


In [ ]:
# Sample documents per cluster + crosstab vs true topic
# YOUR CODE HERE


## Step 10: Visualize in 2D

**Task:** project SVD vectors to 2D with `PCA(n_components=2)`. Plot side-by-side scatters:
- left: colored by **predicted cluster**
- right: colored by **true topic**

Same regions of space should match — visual proof your unsupervised clusters approximate the held-out truth.

In [ ]:
# YOUR CODE HERE


## Step 11: Build the full Pipeline

**Task:** build a single `Pipeline` with these steps in order:
1. `TextCleaner()`
2. `TfidfVectorizer(...)` (same params as Step 5)
3. `TruncatedSVD(n_components=100)`
4. `Normalizer()`
5. `KMeans(n_clusters=K)`

Fit it on the raw `df['text']` list. Compute ARI vs true labels — should match the result from Step 8.

**Why now?** A single artifact that takes raw text in and returns cluster id is what you actually deploy.

In [ ]:
# YOUR CODE HERE


## Step 12: Save the artifact

**Task:**
1. Make folder `artifacts_unsupervised/`.
2. Save the full pipeline with both pickle and joblib.
3. Build a `cluster_summary` dict — for each cluster id store its `top_terms` (top 12) and `size`.
4. Save `metadata.json` with: model name/version/timestamp, `n_clusters`, `n_documents`, metrics (silhouette, ARI, NMI), `cluster_summary`, env versions.

In [ ]:
# YOUR CODE HERE


## Step 13: Reload + assign cluster ids to new docs

**Task:** load `cluster_model.joblib`. Predict cluster ids for these brand-new sentences:
- `'NASA announced a new mission to explore Europa, Jupiter\'s icy moon.'`
- `'The Yankees won last night with a walk-off home run in the ninth.'`
- `'I\'m using OpenGL to render a 3D mesh and the texture mapping looks off.'`
- `'Negotiations between the two governments continue with no clear resolution.'`

For each, print the assigned cluster id and the top 6 terms of that cluster (read from your saved metadata).

**Sanity check:** each sentence should land in a cluster whose top terms match its topic.

In [ ]:
# YOUR CODE HERE


---
## Self-check questions

1. Why did you reduce TF-IDF to 100D before clustering instead of clustering on raw TF-IDF?
2. K-Means gives the highest ARI but DBSCAN found a different number of clusters and lots of noise. Which output do you trust, and on what basis?
3. You don't have ground-truth labels for your real problem. List 3 ways to convince a stakeholder your clusters are meaningful.
4. A new document's prediction looks wrong. The pipeline contains 5 stages — describe how you'd debug to find which stage went sideways.
5. Your corpus grows from 5k docs to 5M docs. Which stages stay the same and which need to change? (Hint: think MiniBatchKMeans, hashing, chunking.)

**YOUR ANSWERS**
1. 
2. 
3. 
4. 
5. 